In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType
import traceback

schemas = {
    "branches": StructType([
        StructField("branch_id", StringType(), False),
        StructField("branch_name", StringType(), True),
        StructField("region", StringType(), True),
        StructField("state", StringType(), True),
        StructField("cost_center_code", StringType(), True),
        StructField("branch_manager_emp_id", StringType(), True),
    ]),
    "employees": StructType([
        StructField("employee_id", StringType(), False),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("branch_id", StringType(), True),
        StructField("department", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("hourly_rate", DoubleType(), True),
        StructField("hire_date", StringType(), True),
        StructField("status", StringType(), True),
        StructField("termination_date", StringType(), True),
    ]),
    "courses": StructType([
        StructField("course_id", StringType(), False),
        StructField("course_name", StringType(), True),
        StructField("category", StringType(), True),
        StructField("delivery_method", StringType(), True),
        StructField("duration_hours", DoubleType(), True),
        StructField("is_mandatory", BooleanType(), True),
    ]),
    "sessions": StructType([
        StructField("session_id", StringType(), False),
        StructField("course_id", StringType(), True),
        StructField("start_timestamp", StringType(), True),
        StructField("facility_cost", DoubleType(), True),
        StructField("max_capacity", IntegerType(), True),
    ]),
    "enrollments": StructType([
        StructField("enrollment_id", StringType(), False),
        StructField("session_id", StringType(), True),
        StructField("employee_id", StringType(), True),
        StructField("status", StringType(), True),
    ]),
    "evaluations": StructType([
        StructField("eval_id", StringType(), False),
        StructField("enrollment_id", StringType(), True),
        StructField("knowledge_test_score", DoubleType(), True),
        StructField("nps_score", IntegerType(), True),
    ]),
    "certifications": StructType([
        StructField("cert_id", StringType(), False),
        StructField("employee_id", StringType(), True),
        StructField("course_id", StringType(), True),
        StructField("license_name", StringType(), True),
        StructField("issue_date", StringType(), True),
        StructField("expiration_date", StringType(), True),
        StructField("status", StringType(), True),
    ]),
    "monthly_performance": StructType([
        StructField("perf_id", StringType(), False),
        StructField("employee_id", StringType(), True),
        StructField("report_month", StringType(), True),
        StructField("transaction_error_rate", DoubleType(), True),
        StructField("cross_sell_ratio", DoubleType(), True),
        StructField("customer_sat_score", DoubleType(), True),
    ]),
}



StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
entity_config = {
    "branches":            {"pk": ["branch_id"],      "date_cols": [],                              "ts_cols": []},
    "employees":           {"pk": ["employee_id"],    "date_cols": ["hire_date","termination_date"],"ts_cols": []},
    "courses":             {"pk": ["course_id"],      "date_cols": [],                              "ts_cols": []},
    "sessions":            {"pk": ["session_id"],     "date_cols": [],                              "ts_cols": ["start_timestamp"]},
    "enrollments":         {"pk": ["enrollment_id"],  "date_cols": [],                              "ts_cols": []},
    "evaluations":         {"pk": ["eval_id"],        "date_cols": [],                              "ts_cols": []},
    "certifications":      {"pk": ["cert_id"],        "date_cols": ["issue_date","expiration_date"],"ts_cols": []},
    "monthly_performance": {"pk": ["perf_id"],        "date_cols": ["report_month"],                "ts_cols": []},
}


In [ ]:
def process_entity(entity_name):
    cfg = entity_config[entity_name]
    bronze_path = f"Files/bronze/raw_json/{entity_name}/{entity_name}.json"
    silver_table = f"silver_{entity_name}"

    try:
        # Bronze files are a single JSON array per file -> multiLine is required,
        # otherwise Spark reads them as JSON-Lines and silently nulls every row.
        df = spark.read.schema(schemas[entity_name]).option("multiLine", True).json(bronze_path)
        raw_count = df.count()

        # Parse date/timestamp strings explicitly rather than trusting inference
        for c in cfg["date_cols"]:
            df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))
        for c in cfg["ts_cols"]:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))  # no format string: tolerates Faker's optional fractional seconds

        # Normalize blank strings to true nulls, trim whitespace
        for f in schemas[entity_name].fields:
            if isinstance(f.dataType, StringType):
                df = df.withColumn(f.name, F.when(F.trim(F.col(f.name)) == "", None).otherwise(F.trim(F.col(f.name))))

        # Deduplicate on natural primary key
        df_clean = df.dropDuplicates(cfg["pk"])
        clean_count = df_clean.count()

        # Lineage columns — standard Silver-layer audit practice
        df_clean = (df_clean
                    .withColumn("_silver_loaded_at", F.current_timestamp())
                    .withColumn("_source_system", F.lit("phcore_api")))

        df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

        print(f"[OK]   {entity_name:<20} raw={raw_count:<6} clean={clean_count:<6} duplicates_removed={raw_count - clean_count}")
        return {"entity": entity_name, "status": "success", "raw_rows": raw_count, "clean_rows": clean_count, "duplicates_dropped": raw_count - clean_count, "error": None}

    except Exception as e:
        print(f"[FAIL] {entity_name:<20} {e}")
        traceback.print_exc()
        return {"entity": entity_name, "status": "failed", "raw_rows": None, "clean_rows": None, "duplicates_dropped": None, "error": str(e)}


In [ ]:
# Runs all 8 entities and produce a data-quality summary
results = [process_entity(name) for name in schemas.keys()]
display(spark.createDataFrame(results))

In [ ]:
print("session check")


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.